In [ ]:
#Project available at https://github.com/PauliusMVGTU/rap-bot

In [ ]:
!pip install pandas numpy tensorflow gTTS

import pandas as pd
import numpy as np
import re
import string
import tensorflow as tf
import pickle
import os

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from gtts import gTTS
from IPython.display import Audio, display

from google.colab import drive
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/rap_model.h5"
TOKENIZER_PATH = "/content/drive/MyDrive/tokenizer.pkl"
SEED_PATH = "/content/drive/MyDrive/shared_seed.txt"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 5.8 MB/s eta 0:00:00


In [ ]:
if os.path.exists(MODEL_PATH) and os.path.exists(TOKENIZER_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(TOKENIZER_PATH, "rb") as f:
        tokenizer = pickle.load(f)

    print("Model and tokenizer loaded from drive")
else:
    print("No saved model found — train first")

✅ Model and tokenizer loaded from Drive


In [ ]:
#Load the dataset
df = pd.read_csv('/content/lyrics_raw.csv')

#Sample a smaller portion of the dataset to reduce memory usage
df = df.sample(frac=0.5, random_state=42).reset_index(drop=True)

#Display the first few rows
display(df.head())

,track_name,artist,raw_lyrics,artist_verses
0,High School,Nicki Minaj,"He said he came from Jamaica, he owned a coupl...","He said he came from Jamaica, he owned a coupl..."
1,OooWee,Rapsody,Yes Lawd! Ooowee Ooowee [Hook: Anderson .Paak...,"Fresh out the bed in my slippers, all my nigga..."
2,Search & Rescue,Drake,"(I-I'm) SADPONY Ayy (I-I'm), yeah BNYX [Choru...","(I-I'm)\nSADPONY\nAyy (I-I'm), yeah\nBNYX\nI n..."
3,They Don't Give A F**** About Us,2Pac,Y'all ain't never just tripped and pictured An...,Y'all ain't never just tripped and pictured\nA...
4,Smile,2Pac,There's gon' be some stuff you gon' see That's...,There's gon' be some stuff you gon' see\nThat'...


## Data Preprocessing

### Subtask:
Clean and prepare the text data for training.

In [ ]:
def preprocess_text(text):
    #Convert text to lowercase
    text = text.lower()
    #Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    #Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_verses'] = df['artist_verses'].apply(preprocess_text)

df = df[df['cleaned_verses'].str.len() > 30]
df = df[df['cleaned_verses'].str.split().str.len() > 8]

display(df[['artist_verses', 'cleaned_verses']].head())

,artist_verses,cleaned_verses
0,"He said he came from Jamaica, he owned a coupl...",he said he came from jamaica he owned a couple...
1,"Fresh out the bed in my slippers, all my nigga...",fresh out the bed in my slippers all my niggas...
2,"(I-I'm)\nSADPONY\nAyy (I-I'm), yeah\nBNYX\nI n...",iim sadpony ayy iim yeah bnyx i need someone t...
3,Y'all ain't never just tripped and pictured\nA...,yall aint never just tripped and pictured and ...
4,There's gon' be some stuff you gon' see\nThat'...,theres gon be some stuff you gon see thats gon...


In [ ]:
tokenizer = Tokenizer(num_words=8000)
tokenizer.fit_on_texts(df['cleaned_verses'])
total_words = min(8000, len(tokenizer.word_index) + 1)
print("Total words:", total_words)

input_sequences = []

for line in df['cleaned_verses']:
    token_list = tokenizer.texts_to_sequences([line])[0][:50000]
    for i in range(1, len(token_list)):
        n_gram = token_list[:i+1]
        input_sequences.append(n_gram)

input_sequences = input_sequences[:50000]

max_sequence_len = max(len(x) for x in input_sequences)
print("Max sequence length:", max_sequence_len)

padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

xs = padded_sequences[:, :-1]
labels = padded_sequences[:, -1]
ys = to_categorical(labels, num_classes=total_words)

print("Training examples:", xs.shape[0])

Total words: 8000
Max sequence length: 2300
Training examples: 50000


## Model Training

### Subtask:
Prepare data for the LSTM model and train it.

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 64))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(128))
model.add(Dropout(0.2))
model.add(Dense(total_words, activation='softmax'))

#Use a safer learning rate
model.compile(
    loss='categorical_crossentropy',
    optimizer=Adam(learning_rate=0.001),
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    xs, ys,
    epochs=15,
    batch_size=256,
    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 87s 404ms/step - accuracy: 0.0345 - loss: 7.4585
Epoch 2/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 79s 405ms/step - accuracy: 0.0352 - loss: 6.5989
Epoch 3/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 84s 417ms/step - accuracy: 0.0352 - loss: 6.4360
Epoch 4/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 82s 420ms/step - accuracy: 0.0385 - loss: 6.3313
Epoch 5/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 82s 420ms/step - accuracy: 0.0480 - loss: 6.1691
Epoch 6/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 421ms/step - accuracy: 0.0565 - loss: 6.0420
Epoch 7/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 423ms/step - accuracy: 0.0604 - loss: 5.9404
Epoch 8/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 421ms/step - accuracy: 0.0684 - loss: 5.8307
Epoch 9/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 421ms/step - accuracy: 0.0758 - loss: 5.7441
Epoch 10/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 143s 425ms/step - accuracy: 0.0809 - loss: 5.6738
Epoch 11/15
196/196 ━━━━━━━━━━━━━━━━━━━━ 83s 425ms/step - accuracy: 0.0864 - loss: 5.5987
Epoch 12/15
196/19

In [ ]:
model.save(MODEL_PATH)

with open(TOKENIZER_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

print("Model and tokenizer saved to drive")

✅ Model and tokenizer saved to Drive


In [ ]:
def sample_with_temperature(preds, temperature=0.85):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-9) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

In [ ]:
def generate_rap(seed_text, next_words=40, temperature=0.85):

    for i in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')

        predicted = model.predict(token_list, verbose=0)[0]
        predicted_index = sample_with_temperature(predicted, temperature)

        word = None
        for w, index in tokenizer.word_index.items():
            if index == predicted_index:
                word = w
                break

        if word is None:
            continue

        #Line breaks every 6 words (rap bars)
        if i % 6 == 0 and i != 0:
            seed_text += "\n" + word
        else:
            seed_text += " " + word

    return seed_text


In [ ]:
def speak(text):
    tts = gTTS(text)
    tts.save("rap.mp3")
    display(Audio("rap.mp3", autoplay=True))

In [ ]:
if os.path.exists(SEED_PATH):
    with open(SEED_PATH, "r", encoding="utf-8") as f:
        lines = f.read().splitlines()

    seed_text = lines[-1] if lines else "i love you"
else:
    seed_text = "i love you"

print("Seed text:", seed_text)

generated_text = generate_rap(seed_text, next_words=60, temperature=0.85)

print("\nGenerated text:")
print(generated_text)

speak(generated_text)

#Get only last line
last_line = generated_text.strip().split("\n")[-1]

#Save it back to drive (so other Colab can read it)
with open(SEED_PATH, "w", encoding="utf-8") as f:
    f.write(last_line)

print("\nBot replied:\n", last_line)


#prompt = "i love you"

#rap = generate_rap(prompt, next_words=48, temperature=0.85)

#print("RAP OUTPUT:\n")
#print(rap)

#speak(rap)

🎤 Seed text: thats i like and blow two

🎤 Generated text:
thats i like and blow two now i do we dont wouldnt
get on you a rawest thinkin
with me i this see so
all and now you go ate
in the hand about my name
i say he business i gon
take it on ya supposed a
club stressin between on the sound
me the need to youre long
they got no im but but



✅ Bot replied:
 they got no im but but
